In [1]:
import torch
print(f"사용 가능 여부: {torch.cuda.is_available()}")
print(f"사용 중인 장치: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(torch.__version__)

import sys
print(sys.executable)

사용 가능 여부: True
사용 중인 장치: NVIDIA A40
2.11.0+cu126
/home/user3/.conda/envs/sjy/bin/python


In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
import os, random, json
from PIL import Image, ImageOps
from collections import Counter
from scipy.stats import mode
from skimage import measure

In [3]:
fix_label_load = pd.read_csv('/workspace/데이터셋/[노션] 3차 프로젝트(4조)-오분류 감정 라벨링.csv')
fix_label = fix_label_load.iloc[:,:5]
fix_label.tail()

,파일명,trn/val,이전 감정,정정 감정,BOX 이상
2114,vmi7fef28ef05e34f34d825fff8344d8879c88383e7d9b...,trn,anger,panic,No
2115,n7jlc06aa011e1601774ff620c4c7411f7c130bc670ee0...,trn,anger,blank,No
2116,rsfd568f8c608f3f1e7ae1a13a8e8788a1f02783ae1932...,trn,anger,sadness,No
2117,z17b5a1ff8a235cec1a1aaf585a8d63dc1a043d314ffd8...,trn,anger,sadness,No
2118,4vgo5cbf032f9cb936d955907cd5c5f6b94360486f87b5...,trn,anger,anger,No


In [4]:
change_label = fix_label[~(fix_label['이전 감정']==fix_label['정정 감정'])]
box_error_img = fix_label[fix_label['BOX 이상']=='Yes']

In [5]:
ADRESS = '/workspace/데이터셋'

img_adress = f'{ADRESS}/img'
label_adress = f'{ADRESS}/label'
segmentation_adress = f'{ADRESS}/segmentation'

In [6]:
EMOTIONS = ['anger', 'happy', 'sadness', 'panic']
DATA_TYPES_TO_PROCESS = ['train', 'val']

In [7]:
def load_emotion_data_files(emotion, data_type):
    """
    주어진 감정과 데이터 유형에 대한 json 라벨과 npz 세그먼트를 불러오는 함수.
    (json_data, npz_data)가 디폴트 반환값이지만 데이터를 찾을 수 없는 경우 (None, None)을 반환
    """
    json_file_path = os.path.join(label_adress, data_type, f'{data_type}_{emotion}.json')
    npz_file_path = os.path.join(segmentation_adress, data_type, f'{data_type}_{emotion}.npz')

    current_json_data = None
    current_npz_data = None

    # json 데이터 로드
    if os.path.exists(json_file_path):
        try:
            with open(json_file_path, 'r', encoding='cp949') as f:
                current_json_data = json.load(f)
            print(f"Loaded JSON for {emotion} ({data_type}) from {json_file_path}")
        except Exception as e:
            print(f"Error loading JSON for {emotion} ({data_type}) from {json_file_path}: {e}")
    else:
        print(f"JSON file not found: {json_file_path}")

    # npz 데이터 로드
    if os.path.exists(npz_file_path):
        try:
            current_npz_data = np.load(npz_file_path)
            print(f"Loaded NPZ for {emotion} ({data_type}) from {npz_file_path}")
        except Exception as e:
            print(f"Error loading NPZ for {emotion} ({data_type}) from {npz_file_path}: {e}")
    else:
        print(f"NPZ file not found: {npz_file_path}")

    return current_json_data, current_npz_data

In [8]:
def combined_data_iterator():
    """
    이미지, 라벨, 세그먼트 데이터를 매칭시켜주는 함수
    EMOTIONS, DATA_TYPES_TO_PROCESS 변수로 지정된 데이터 유형과 감정에 대해서만 조회.
    """
    for data_type in DATA_TYPES_TO_PROCESS:
        for emotion in EMOTIONS:
            json_labels, npz_segments = load_emotion_data_files(emotion, data_type)

            # 라벨, 세그먼트 중에 하나라도 누락되어있을 경우 스킵.
            if json_labels is None or npz_segments is None:
                print(f"Skipping {emotion} {data_type} due to missing JSON or NPZ data.")
                if npz_segments is not None: npz_segments.close()
                continue

            # img 데이터 로드
            img_emotion_dir = os.path.join(img_adress, data_type, emotion)

            # 이미지가 누락되어있을 경우 스킵.
            if not os.path.exists(img_emotion_dir):
                print(f"Image directory not found: {img_emotion_dir}. Skipping this emotion for {data_type}.")
                if npz_segments is not None: npz_segments.close()
                continue

            for json_entry in json_labels:
                filename = json_entry.get('filename')
                if not filename: continue # 이미지 파일 이름이 없으면 스킵.
                
                # 재분류된 이미지 라벨 정정
                current_emotion = emotion
                if change_label.파일명.str.contains(filename.split('.')[0]).any():
                    current_emotion = change_label.loc[change_label.파일명.str.contains(filename.split('.')[0]),'정정 감정'].iloc[0]

                boxes_err = False
                if box_error_img.파일명.str.contains(filename.split('.')[0]).any(): boxes_err = True

                img_full_path = os.path.join(img_emotion_dir, filename)
                segmentation_mask = npz_segments.get(filename)

                # 파일 매칭. 이미지 + 라벨 + 세그먼트
                if os.path.exists(img_full_path) and segmentation_mask is not None:
                    entry_data = {
                        'filename': filename,
                        'data_type': data_type,
                        'gender': json_entry.get('gender'),
                        'age': json_entry.get('age'),
                        'img_path': img_full_path,
                        'faceExp': current_emotion,
                        'segmentation_mask': segmentation_mask,
                      # 'isProf': json_entry.get('isProf'),
                      # 'faceExp_uploader': json_entry.get('faceExp_uploader'),
                      # 'bg_uploader': json_entry.get('bg_uploader'),
                    }

                    min_x_vals, min_y_vals, max_x_vals, max_y_vals = [], [], [], []
                    for annotator in ['A', 'B', 'C']:
                        annot_key = f'annot_{annotator}'
                        if annot_key in json_entry and isinstance(json_entry[annot_key], dict):
                            boxes = json_entry[annot_key].get('boxes')
                            if boxes and all(k in boxes for k in ['minX', 'minY', 'maxX', 'maxY']):
                                min_x_vals.append(boxes['minX'])
                                min_y_vals.append(boxes['minY'])
                                max_x_vals.append(boxes['maxX'])
                                max_y_vals.append(boxes['maxY'])

                    avg_min_x, avg_min_y, avg_max_x, avg_max_y = None, None, None, None
                    if min_x_vals:
                        avg_min_x = np.mean(min_x_vals).item()
                        avg_min_y = np.mean(min_y_vals).item()
                        avg_max_x = np.mean(max_x_vals).item()
                        avg_max_y = np.mean(max_y_vals).item()
                        entry_data['boxes'] = {'maxX': avg_min_x, 'maxY': avg_min_y, 'minX': avg_max_x, 'minY': avg_max_y}
                        entry_data['boxes_err'] = boxes_err

                    yield entry_data

            if npz_segments is not None: npz_segments.close()

In [9]:
all_matched_data_gen = combined_data_iterator()

In [10]:
next(all_matched_data_gen)

Loaded JSON for anger (train) from /workspace/데이터셋/label/train/train_anger.json
Loaded NPZ for anger (train) from /workspace/데이터셋/segmentation/train/train_anger.npz


{'filename': '6oj439e3fbcc52759fb3093035b7c0ecc55c93543dae63ec1162f5f6bfc6e481l.jpg',
 'data_type': 'train',
 'gender': '남',
 'age': 30,
 'img_path': '/workspace/데이터셋/img/train/anger/6oj439e3fbcc52759fb3093035b7c0ecc55c93543dae63ec1162f5f6bfc6e481l.jpg',
 'faceExp': 'blank',
 'segmentation_mask': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 4, 4, 4],
        [0, 0, 0, ..., 4, 4, 4],
        [0, 0, 0, ..., 4, 4, 4]], shape=(2208, 2944), dtype=uint8),
 'boxes': {'maxX': 1339.8584383333334,
  'maxY': 531.096642,
  'minX': 2073.368938333333,
  'minY': 1488.0557753333333},
 'boxes_err': False}

In [11]:
def mask_to_yolo_segmentation(mask, image_width, image_height, emotion_label, emotion_to_yolo_id_map):
    """
    face (레이블 3) 세그먼트 마스크를 YOLOv8 세그먼트 형식으로 변환.
    emotion_label을 YOLO 클래스 id로 사용.
    face 세그먼트 영역만 처리.

    Args:
        mask (np.array): 현재 세그먼트 마스크. (정수 레이블)
        image_width (int): 원본 이미지의 너비.
        image_height (int): 원본 이미지의 높이.
        emotion_label (str): 현재 감정 레이블. (anger, happy, sadness, panic, blank)
        emotion_to_yolo_id_map (dict): 감정 레이블을 YOLO 클래스 ID로 매핑

    Returns:
        list: 각 얼굴 객체당 하나씩, YOLOv8 형식의 세그먼트 문자열 리스트
    """
    yolo_labels = []
    face_label_id = 3  # 세그먼테이션 레이블에서 {0: background, 1: 'hair',  2: 'body',  3: 'face',  4: 'cloth', 5: 'etc'}
    yolo_class_id = emotion_to_yolo_id_map.get(emotion_label)
    if yolo_class_id is None: return []

    # face 레이블에 해당하는 윤곽선 계산.
    # 'mask == face_label_id'는 face 객체에 대한 이진 마스크 생성.
    contours = measure.find_contours(mask == face_label_id, 0.5) # 0.5 값은 등고선.

    for contour in contours:
        # YOLOv8은 x, y 쌍 포맷을 기대함. -> 윤곽선 평탄화 및 정규화 진행.
        # skimage 윤곽선은 (row, col) 즉. (y, x) 포맷.
        contour_normalized = []
        if len(contour) < 3: continue # 윤곽은 다각형으로 그려질텐데 그렇담, 다각형이 그려질려면 최소 3개의 점이 필요함.

        for y, x in contour: # 좌표를 [0, 1] 범위로 정규화
            x_norm = x / image_width
            y_norm = y / image_height
            contour_normalized.extend([x_norm, y_norm])

        # 좌표를 공백으로 연결하고 class_id를 접두사로 붙임.
        if len(contour_normalized) > 0:
            yolo_line = f"{yolo_class_id} {' '.join(map(str, contour_normalized))}"
            yolo_labels.append(yolo_line)
    return yolo_labels

In [12]:
DATASET_ROOT = '/workspace/user3/yolov8_dataset-2'

SUBDIRECTORIES = [
    'images/train',
    'images/val',
    'labels/train',
    'labels/val']

for subdir in SUBDIRECTORIES:
    path = os.path.join(DATASET_ROOT, subdir)
    os.makedirs(path, exist_ok=True)

In [13]:
import yaml

# --- YOLOv8에 대한 감정 클래스 매핑 및 이름 정의 ---
# 감정 리스트를 기반으로 한 표준 클래스 이름. data.yaml 파일의 'name' 부분
EMOTION_CANONICAL_NAMES = ['blank', 'anger', 'happy', 'sadness', 'panic']
emotion_to_yolo_class_id = {'blank': 0, 'anger': 1, 'happy': 2, 'sadness': 3, 'panic': 4}

# data.yaml 파일 내용 생성
data_yaml_content = {
    'path': DATASET_ROOT, # 데이터셋 루트 경로
    'train': 'images/train',  # 학습 이미지 상대 경로
    'val': 'images/val',    # 검증 이미지 상대 경로
    'nc': len(EMOTION_CANONICAL_NAMES), # 감정 클래스 수
    'names': EMOTION_CANONICAL_NAMES    # 감정 클래스 이름 목록
}

# data.yaml 파일 저장 경로
data_yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')

# data.yaml 파일 작성
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"data.yaml 파일이 '{data_yaml_path}'에 성공적으로 생성되었습니다.")
print("생성된 data.yaml 내용:")
with open(data_yaml_path, 'r') as f:
    print(f.read())

data.yaml 파일이 '/workspace/user3/yolov8_dataset-2/data.yaml'에 성공적으로 생성되었습니다.
생성된 data.yaml 내용:
path: /workspace/user3/yolov8_dataset-2
train: images/train
val: images/val
nc: 5
names:
- blank
- anger
- happy
- sadness
- panic



In [14]:
full_dataset_iterator = combined_data_iterator()

def process_and_save_dataset(iterator, dataset_root, emotion_map):
    """
    Combined_data_iterator에서 데이터를 가져와 YOLOv8 감정 세그멘테이션 형식으로 저장.
    이미지는 images/train/ 또는 images/val/ 에, 라벨은 labels/train/ 또는 labels/val/ 에 저장.
    라벨은 face 영역에 대한 감정 클래스 id를 사용.
    """
    processed_count = 0
    skipped_count = 0

    for sample_entry in iterator:
        try:
            filename = sample_entry['filename']
            data_type = sample_entry['data_type'] # 'train' or 'val'
            original_img_path = sample_entry['img_path']
            segmentation_mask = sample_entry['segmentation_mask']
            emotion_label = sample_entry['faceExp']

            # --- 1. 이미지 저장 ---
            img = Image.open(original_img_path).convert('RGB')
            img = ImageOps.exif_transpose(img)
            img_width, img_height = img.size

            # 저장 경로 설정
            img_output_dir = os.path.join(dataset_root, 'images', data_type)
            img_output_path = os.path.join(img_output_dir, filename)
            
            # 파일 이름에서 확장자 제거 (label 파일과 맞추기 위함)
            base_filename = os.path.splitext(filename)[0]
            
            # 이미지 파일 저장
            os.makedirs(img_output_dir, exist_ok=True)
            img.save(img_output_path)

            # --- 2. YOLOv8 라벨 생성 및 저장 ---
            # segmentation_mask와 class_mapping을 사용하여 YOLOv8 라벨 생성
            yolo_segmentation_labels = mask_to_yolo_segmentation(
                segmentation_mask, img_width, img_height, emotion_label, emotion_map
            )

            # 라벨 저장 경로 설정
            label_output_dir = os.path.join(dataset_root, 'labels', data_type)
            label_output_path = os.path.join(label_output_dir, f'{base_filename}.txt')
            
            os.makedirs(label_output_dir, exist_ok=True)

            if yolo_segmentation_labels:
                with open(label_output_path, 'w') as f:
                    for line in yolo_segmentation_labels:
                        f.write(line + '\n')
                processed_count += 1
            else:
                print(f"Warning: No YOLO emotion segmentation labels generated for {filename} (Emotion: {emotion_label}). Skipping label file creation.")
                skipped_count += 1

        except Exception as e:
            print(f"Error processing {sample_entry.get('filename', 'Unknown file')} (Emotion: {emotion_label if 'emotion_label' in locals() else 'N/A'}): {e}")
            skipped_count += 1

    print(f"\nDataset processing complete. Processed {processed_count} samples, skipped {skipped_count} samples.")

# 데이터셋 처리 및 저장 시작
print("Starting dataset preparation for YOLOv8 emotion segmentation training...")
process_and_save_dataset(full_dataset_iterator, DATASET_ROOT, emotion_to_yolo_class_id)
print("Dataset preparation finished.")

data_yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')
with open(data_yaml_path, 'r') as f:
    print(f"\nContent of {data_yaml_path}:\n{f.read()}")

Starting dataset preparation for YOLOv8 emotion segmentation training...
Loaded JSON for anger (train) from /workspace/데이터셋/label/train/train_anger.json
Loaded NPZ for anger (train) from /workspace/데이터셋/segmentation/train/train_anger.npz


Loaded JSON for happy (train) from /workspace/데이터셋/label/train/train_happy.json
Loaded NPZ for happy (train) from /workspace/데이터셋/segmentation/train/train_happy.npz
Loaded JSON for sadness (train) from /workspace/데이터셋/label/train/train_sadness.json
Loaded NPZ for sadness (train) from /workspace/데이터셋/segmentation/train/train_sadness.npz
Loaded JSON for panic (train) from /workspace/데이터셋/label/train/train_panic.json
Loaded NPZ for panic (train) from /workspace/데이터셋/segmentation/train/train_panic.npz
Loaded JSON for anger (val) from /workspace/데이터셋/label/val/val_anger.json
Loaded NPZ for anger (val) from /workspace/데이터셋/segmentation/val/val_anger.npz
Loaded JSON for happy (val) from /workspace/데이터셋/label/val/val_happy.json
Loaded NPZ for happy (val) from /workspace/데이터셋/segmentation/val/val_happy.npz
Loaded JSON for sadness (val) from /workspace/데이터셋/label/val/val_sadness.json
Loaded NPZ for sadness (val) from /workspace/데이터셋/segmentation/val/val_sadness.npz
Loaded JSON for panic (val) fr

- CPU로 돌아가고 있었다;;

In [ ]:
from ultralytics import YOLO

data_yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')

model = YOLO('yolov8n-seg.pt') # segmentation 모델이므로 '-seg' 접미사가 붙은 모델을 사용.

print("YOLOv8 모델 학습을 시작합니다...")
results = model.train(data=data_yaml_path, epochs=100, imgsz=640, batch=16, name='yolov8n_custom_seg_model-2')

print("\n--- 학습 완료 --- ")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/user3/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8 모델 학습을 시작합니다...
Ultralytics 8.4.38 🚀 Python-3.10.20 torch-2.11.0+cu130 CPU (Intel Xeon Gold 6342 CPU @ 2.80GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/user3/yolov8_dataset-2/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

/home/user3/.conda/envs/sjy/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Overriding model.yaml nc=80 with nc=5

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

/home/user3/.conda/envs/sjy/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


val: Scanning /workspace/user3/yolov8_dataset-2/labels/val... 1200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1200/1200 74.5it/s 16.1s0.1s
val: New cache created: /workspace/user3/yolov8_dataset-2/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001111, momentum=0.9) with parameter groups 66 weight(decay=0.0), 77 weight(decay=0.0005), 76 bias(decay=0.0)
Plotting labels to /workspace/user3/runs/segment/yolov8n_custom_seg_model/labels.jpg... 
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /workspace/user3/runs/segment/yolov8n_custom_seg_model
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/100         0G     0.6926      1.438      2.209     0.9873          0         21        640: 100% ━━━━━━━━━━━━ 375/375 3.6s/it 22:

KeyboardInterrupt: 

- 기왕 다시 돌리는 김에 YOLO26 모델로 재시작

In [ ]:
from ultralytics import YOLO

data_yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')

model = YOLO('yolo26s-seg.pt') # segmentation 모델이므로 '-seg' 접미사가 붙은 모델을 사용.

print("YOLO26 모델 학습을 시작합니다...")
results = model.train(data=data_yaml_path, epochs=100, imgsz=640, batch=16, device=0, name='yolo26s_custom_seg_model')

print("\n--- 학습 완료 --- ")

YOLO26 모델 학습을 시작합니다...
New https://pypi.org/project/ultralytics/8.4.39 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.38 🚀 Python-3.10.20 torch-2.11.0+cu126 CUDA:0 (NVIDIA A40, 45515MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/user3/yolov8_dataset-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-seg.pt, momentum=0

In [21]:
model.val()

Ultralytics 8.4.38 🚀 Python-3.10.20 torch-2.11.0+cu126 CUDA:0 (NVIDIA A40, 45515MiB)
YOLO26s-seg summary (fused): 139 layers, 10,367,275 parameters, 0 gradients, 34.1 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.1 ms, read: 75.1±41.7 MB/s, size: 873.7 KB)
val: Scanning /workspace/user3/yolov8_dataset-2/labels/val.cache... 1200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1200/1200 68.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 75/75 6.7it/s 11.1s0.1s
                   all       1200       4419      0.551      0.262      0.254      0.223      0.543      0.265      0.267      0.226
                 anger        129        480       0.26      0.258      0.131      0.112      0.253      0.258      0.135      0.111
                 happy        187        739      0.455      0.207      0.202       0.18      0.454      0.212      0.216      0.185
               sadn

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8e2523a470>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,  

emotion_to_yolo_class_id = {'anger': 1, 'happy': 2, 'sadness': 3, 'panic': 4, 'blank': 0}   # 변경 전
- 혹여나 문제될까 싶었는데 역시나 happy id가 1이 되어버렸다.
- 다시 blank를 맨 앞으로 옮겨주면서 재시작, 동시에 증강 파라미터를 추가로 약간 건드렸다. + 러닝레이트도 0.1에서 0.001로 변경.

emotion_to_yolo_class_id = {'blank': 0, 'anger': 1, 'happy': 2, 'sadness': 3, 'panic': 4}   # 변경 후

In [15]:
from ultralytics import YOLO

data_yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')

model = YOLO('yolo26s-seg.pt') # segmentation 모델이므로 '-seg' 접미사가 붙은 모델을 사용.

print("YOLO26 모델 학습을 시작합니다...")
results = model.train(data=data_yaml_path, epochs=100, imgsz=1024, batch=16, device=0, cls=2.0, degrees=10.0, mosaic=1.0, mixup=0.2, optimizer='AdamW', lr0=0.001, name='yolo26s_custom_seg_model')

print("\n--- 학습 완료 --- ")

YOLO26 모델 학습을 시작합니다...
New https://pypi.org/project/ultralytics/8.4.39 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.38 🚀 Python-3.10.20 torch-2.11.0+cu126 CUDA:0 (NVIDIA A40, 45515MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/user3/yolov8_dataset-2/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo26s-seg.pt, momentu

In [16]:
TRAINED_MODEL_PATH = '/workspace/user3/runs/segment/yolo26s_custom_seg_model2/weights/best.pt'

# 베스트 모델 불러오기.
loaded_model = YOLO(TRAINED_MODEL_PATH)

In [ ]:
_original_data_types_to_process = DATA_TYPES_TO_PROCESS
_original_emotions = EMOTIONS
DATA_TYPES_TO_PROCESS = ['val']
EMOTIONS = ['anger']

temp_val_iterator = combined_data_iterator()

# 변수값 원래대로 복원
DATA_TYPES_TO_PROCESS = _original_data_types_to_process
EMOTIONS = _original_emotions

In [ ]:
temp_val_iterator = next(temp_val_iterator)

In [ ]:
TEST_IMAGE_PATH = temp_val_iterator['img_path']

# 예측 수행
results = loaded_model.predict(source=TEST_IMAGE_PATH, save=True, conf=0.25, iou=0.7)
print(temp_val_iterator['faceExp'])

# 예측 결과 시각화 (첫 번째 이미지에 대한 결과)
for r in results:
    im_array = r.plot()  # plot a BGR numpy array of predictions
    im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image

    plt.figure(figsize=(10, 10))
    plt.imshow(im)
    plt.title(f"Prediction on: {os.path.basename(TEST_IMAGE_PATH)}")
    plt.axis('off')
    plt.show()

    # 예측된 감정 클래스 및 신뢰도 출력
    print("\nPredicted Emotion Segmentation Results:")
    if r.boxes: # Detect (bounding box) results
        for box in r.boxes:
            class_id = int(box.cls[0])
            conf = box.conf[0]
            print(f"  Class ID: {class_id}, Confidence: {conf:.2f}")
    if r.masks: # Segmentation results
        # Masks provide pixel-level segmentation, but class_id and conf come from boxes
        print(f"  Total detected masks: {len(r.masks)}")

    # 각 class_id가 어떤 감정에 매핑되는지 알려줍니다.
    print("\nEmotion Class Mapping (ID -> Name):")
    # EMOTION_CANONICAL_NAMES는 이전에 data.yaml 생성 시 사용된 순서대로 감정 이름을 포함합니다.
    for idx, name in enumerate(EMOTION_CANONICAL_NAMES):
        print(f"  {idx}: {name}")


Loaded JSON for anger (val) from /workspace/데이터셋/label/val/val_anger.json
Loaded NPZ for anger (val) from /workspace/데이터셋/segmentation/val/val_anger.npz
테스트 이미지 경로: /workspace/데이터셋/img/val/anger/7uji1c8ab7778bc286c3cd90077847f9a906f00b210838ba57877a29abd60ffix.jpg



image 1/1 /workspace/데이터셋/img/val/anger/7uji1c8ab7778bc286c3cd90077847f9a906f00b210838ba57877a29abd60ffix.jpg: 768x1024 1 anger, 14.8ms
Speed: 6.7ms preprocess, 14.8ms inference, 1.3ms postprocess per image at shape (1, 3, 768, 1024)
Results saved to /workspace/user3/runs/segment/predict2


<Figure size 1000x1000 with 1 Axes>


Predicted Emotion Segmentation Results:
  Class ID: 1, Confidence: 0.96
  Total detected masks: 1

Emotion Class Mapping (ID -> Name):
  0: blank
  1: anger
  2: happy
  3: sadness
  4: panic


In [33]:
from moviepy.video.io.VideoFileClip import VideoFileClip

original_path = '/workspace/user3/오빠 소리 듣고 리액션 고장 난 덱스 ㅋㅋ #언니네산지직송in칼라페.mp4'
target_path = '/workspace/user3/dex_fixed.mp4'

try:
    clip = VideoFileClip(original_path)
    # temp_audiofile 경로를 지정해주면 권한 오류를 피하기 좋습니다.
    clip.write_videofile(target_path, codec='libx264', audio_codec='aac')
    clip.close()
    print("성공적으로 변환되었습니다!")
except Exception as e:
    print(f"변환 중 오류 발생: {e}")

MoviePy - Building video /workspace/user3/dex_fixed.mp4.
MoviePy - Writing audio in dex_fixedTEMP_MPY_wvf_snd.mp4


chunk:   0%|          | 0/369 [00:00<?, ?it/s, now=None]

MoviePy - Done.
MoviePy - Writing video /workspace/user3/dex_fixed.mp4



frame_index:  99%|█████████▉| 496/501 [00:09<00:00, 86.78it/s, now=None]/home/user3/.conda/envs/sjy/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file /workspace/user3/오빠 소리 듣고 리액션 고장 난 덱스 ㅋㅋ #언니네산지직송in칼라페.mp4, 1969920 bytes wanted but 0 bytes read at frame index 500 (out of a total 501 frames), at time 16.68/16.72 sec. Using the last valid frame instead.
  warnings.warn(


MoviePy - Done !
MoviePy - video ready /workspace/user3/dex_fixed.mp4
성공적으로 변환되었습니다!


In [35]:
# 리스트에 파일 경로를 여러 개 넣으세요
VIDEO_LIST = [
    '/workspace/user3/dex_fixed.mp4',
]

for path in VIDEO_LIST:
    if os.path.exists(path):
        results = loaded_model.predict(source=path, stream=True, save=True, conf=0.25, iou=0.7)
        for r in results: # stream=True 시 필수 실행
            pass



video 1/1 (frame 1/501) /workspace/user3/dex_fixed.mp4: 1024x576 1 blank, 24.4ms
video 1/1 (frame 2/501) /workspace/user3/dex_fixed.mp4: 1024x576 1 blank, 14.9ms
video 1/1 (frame 3/501) /workspace/user3/dex_fixed.mp4: 1024x576 1 blank, 13.1ms
video 1/1 (frame 4/501) /workspace/user3/dex_fixed.mp4: 1024x576 (no detections), 11.9ms
video 1/1 (frame 5/501) /workspace/user3/dex_fixed.mp4: 1024x576 (no detections), 17.5ms
video 1/1 (frame 6/501) /workspace/user3/dex_fixed.mp4: 1024x576 (no detections), 17.9ms
video 1/1 (frame 7/501) /workspace/user3/dex_fixed.mp4: 1024x576 (no detections), 18.4ms
video 1/1 (frame 8/501) /workspace/user3/dex_fixed.mp4: 1024x576 (no detections), 18.0ms
video 1/1 (frame 9/501) /workspace/user3/dex_fixed.mp4: 1024x576 1 blank, 14.2ms
video 1/1 (frame 10/501) /workspace/user3/dex_fixed.mp4: 1024x576 1 blank, 14.0ms
video 1/1 (frame 11/501) /workspace/user3/dex_fixed.mp4: 1024x576 1 blank, 14.6ms
video 1/1 (frame 12/501) /workspace/user3/dex_fixed.mp4: 1024x576 